# Week 14 Optional: Parameter-Efficient Fine-Tuning with LoRA

This optional notebook introduces **LoRA (Low-Rank Adaptation)** — a technique
for fine-tuning models by training only ~1% of the parameters.

**Why LoRA?**
- Full fine-tuning of a 7B model needs ~28GB GPU memory
- LoRA reduces this to ~8GB by only training small adapter matrices
- Quality is often within 1-2% of full fine-tuning

**What you'll learn:**
1. How LoRA works (low-rank decomposition of weight updates)
2. Configure and apply LoRA to Flan-T5
3. Compare parameter counts: full fine-tuning vs LoRA
4. Train and evaluate a LoRA-adapted model
5. Understand QLoRA (4-bit quantization + LoRA)

**Prerequisites**: Completed Week 14 main notebook

**GPU**: T4 GPU recommended (Runtime → Change runtime type → T4 GPU)

In [ ]:
# =============================================================================
# SETUP: Install and Import Libraries
# =============================================================================

!pip install -q transformers peft accelerate bitsandbytes datasets evaluate scikit-learn

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import torch
import numpy as np
import pandas as pd
import evaluate
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Device check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

print("\n✅ Setup complete!")

# Section 1: How LoRA Works

## The Problem with Full Fine-Tuning

In the main notebook, we fine-tuned **all 67M parameters** of DistilBERT. That's
fine for a small model on a T4 GPU. But what about larger models?

| Model | Parameters | Full FT Memory | LoRA Memory |
|-------|-----------|----------------|-------------|
| DistilBERT | 67M | ~1 GB | ~0.3 GB |
| BERT-large | 340M | ~4 GB | ~1 GB |
| Llama-2-7B | 7B | ~28 GB | ~8 GB |
| Llama-2-13B | 13B | ~52 GB | ~16 GB |

## How LoRA Works

Instead of updating a full weight matrix **W** (d × d), LoRA adds two small
matrices **A** (d × r) and **B** (r × d) where r << d:

```
W_new = W_frozen + A × B
```

- **W_frozen**: Original weights (frozen, not updated)
- **A × B**: Low-rank update (only these are trained)
- **r**: Rank (typically 4-16, controls adapter size)

For a 768×768 attention matrix with rank r=8:
- Full fine-tuning: 768 × 768 = **589,824 parameters**
- LoRA: (768 × 8) + (8 × 768) = **12,288 parameters** (2% of original!)

## Key LoRA Hyperparameters

- **r** (rank): Size of adapter. Higher = more capacity but more params. Typical: 4-16
- **lora_alpha**: Scaling factor. Usually set to 2×r
- **target_modules**: Which layers to adapt (usually attention: q, v projections)
- **lora_dropout**: Regularization dropout on adapter layers

In [ ]:
# =============================================================================
# DEMO: Load DistilBERT and Apply LoRA
# =============================================================================

MODEL_NAME = "distilbert-base-uncased"
ID2LABEL = {0: "legitimate", 1: "fraud"}
LABEL2ID = {"legitimate": 0, "fraud": 1}

# Load base model
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label=ID2LABEL, label2id=LABEL2ID
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Count base model parameters
base_params = sum(p.numel() for p in base_model.parameters())
print(f"Base model parameters: {base_params:,}")

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,        # Sequence classification (not seq2seq)
    r=8,                                # Rank of adapter matrices
    lora_alpha=16,                      # Scaling factor (usually 2*r)
    lora_dropout=0.1,                   # Dropout for regularization
    target_modules=["q_lin", "v_lin"],  # DistilBERT attention layer names
)

# Apply LoRA — this freezes the base model and adds adapter layers
lora_model = get_peft_model(base_model, lora_config)

# Compare parameter counts
lora_model.print_trainable_parameters()

# Visual comparison
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in lora_model.parameters() if not p.requires_grad)
print(f"\nTrainable: {trainable:,} ({trainable/base_params:.2%})")
print(f"Frozen:    {frozen:,} ({frozen/base_params:.2%})")

In [ ]:
# =============================================================================
# DEMO: Prepare the Same Fraud Dataset
# =============================================================================
# Re-use the same data pipeline from the main notebook.

# Try loading Week 13 synthetic data, fall back to template data
import os

CSV_PATH = "synthetic_fraud_data.csv"

if os.path.exists(CSV_PATH):
    train_df = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(train_df)} synthetic transactions from Week 13")
else:
    # Template-based fallback (same as main notebook)
    print("Week 13 CSV not found — using template-based training data")
    fraud_templates = [
        "Unauthorized wire transfer of ${amount} to unknown account in {country}",
        "Multiple rapid ATM withdrawals totaling ${amount} across {n} locations",
        "Online purchase of ${amount} from suspicious merchant, shipping to {country}",
        "Account takeover: password changed and ${amount} transferred within minutes",
        "Card-not-present transaction of ${amount} from unrecognized IP address",
    ]
    legit_templates = [
        "Monthly payroll deposit of ${amount} from registered employer",
        "Grocery purchase of ${amount} at local supermarket",
        "Recurring utility payment of ${amount} to electric company",
        "ATM withdrawal of ${amount} at usual branch location",
        "Online subscription renewal of ${amount} for streaming service",
    ]
    import random
    random.seed(42)
    rows = []
    for i in range(50):
        t = random.choice(fraud_templates)
        desc = t.replace("${amount}", str(random.randint(500, 50000)))
        desc = desc.replace("{country}", random.choice(["Nigeria", "Romania", "Russia"]))
        desc = desc.replace("{n}", str(random.randint(3, 8)))
        rows.append({"description": desc, "label": "fraud"})
    for i in range(50):
        t = random.choice(legit_templates)
        desc = t.replace("${amount}", str(random.randint(20, 5000)))
        rows.append({"description": desc, "label": "legitimate"})
    train_df = pd.DataFrame(rows)

print(f"Training data: {len(train_df)} transactions")
print(f"Label distribution:\n{train_df['label'].value_counts().to_string()}")

# Tokenize
def tokenize_fn(examples):
    return tokenizer(examples['description'], truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(
    train_df.rename(columns={'label': 'label_str'}).assign(
        label=train_df['label'].map(LABEL2ID)
    )[['description', 'label']]
)
train_tokenized = train_dataset.map(tokenize_fn, batched=True)
train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(f"\n✅ Dataset ready: {len(train_tokenized)} examples")

In [ ]:
# =============================================================================
# DEMO: Train with LoRA
# =============================================================================

accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

# Split into train/val (80/20)
split_idx = int(len(train_tokenized) * 0.8)
val_tokenized = train_tokenized.select(range(split_idx, len(train_tokenized)))
train_split = train_tokenized.select(range(split_idx))

# Training arguments — same as main notebook
training_args = TrainingArguments(
    output_dir='./fraud-classifier-lora',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-4,                # LoRA typically uses higher LR than full FT
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=10,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

# Create Trainer with the LoRA model
lora_trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_split,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Training LoRA adapter (only ~0.2% of parameters)...")
print("=" * 50)
lora_result = lora_trainer.train()
print("=" * 50)
print(f"\n✅ LoRA training complete!")
print(f"  Training loss: {lora_result.training_loss:.4f}")
print(f"  Training time: {lora_result.metrics['train_runtime']:.0f}s")

In [ ]:
# =============================================================================
# DEMO: Compare Full Fine-Tuning vs LoRA
# =============================================================================

# Evaluate LoRA model on validation set
lora_eval = lora_trainer.evaluate()

print("LoRA Fine-Tuning Results:")
print(f"  Accuracy:  {lora_eval['eval_accuracy']:.1%}")
print(f"  Precision: {lora_eval['eval_precision']:.1%}")
print(f"  Recall:    {lora_eval['eval_recall']:.1%}")
print(f"  F1 Score:  {lora_eval['eval_f1']:.1%}")

# Comparison table
comparison = pd.DataFrame([
    {
        'Method': 'Full Fine-Tuning (main notebook)',
        'Trainable Params': f'{base_params:,} (100%)',
        'Training Time': 'Baseline',
        'Accuracy': 'See main notebook',
    },
    {
        'Method': 'LoRA (r=8)',
        'Trainable Params': f'{trainable:,} ({trainable/base_params:.2%})',
        'Training Time': f'{lora_result.metrics["train_runtime"]:.0f}s',
        'Accuracy': f'{lora_eval["eval_accuracy"]:.1%}',
    },
])
print("\nComparison:")
display(comparison)

print("\nKey takeaway: LoRA trains ~0.2% of parameters with comparable accuracy.")
print("For DistilBERT (67M), the savings are modest. But for a 7B model,")
print("this is the difference between needing an A100 and a T4.")

# Section 2: QLoRA — Quantization + LoRA

**QLoRA** combines two techniques:

1. **4-bit Quantization**: Load the base model in 4-bit precision (NF4 format),
   reducing memory by ~4x compared to FP32
2. **LoRA Adapters**: Train small adapter layers in full precision on top

This makes it possible to fine-tune a **7B parameter model on a single T4 GPU**
(16GB VRAM) — something that would normally require 28GB+ with full fine-tuning.

### BitsAndBytes Configuration

```python
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                  # Load model in 4-bit
    bnb_4bit_quant_type="nf4",          # NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16,  # Compute in FP16
    bnb_4bit_use_double_quant=True,     # Double quantization for more savings
)

model = AutoModel.from_pretrained(
    "model-name",
    quantization_config=bnb_config,
    device_map="auto",
)
```

> **Note**: QLoRA requires a CUDA GPU. It won't work on CPU or Apple Silicon.
> On Colab's free T4, you can QLoRA models up to ~7B parameters.

In [ ]:
# =============================================================================
# DEMO: Memory Comparison — FP32 vs 4-bit
# =============================================================================
# We'll show the memory difference by loading DistilBERT in different precisions.
# (For larger models, the difference is even more dramatic.)

import sys

# Calculate memory for different precisions
params = base_params
fp32_mb = params * 4 / 1e6       # 4 bytes per FP32 param
fp16_mb = params * 2 / 1e6       # 2 bytes per FP16 param
int8_mb = params * 1 / 1e6       # 1 byte per INT8 param
int4_mb = params * 0.5 / 1e6     # 0.5 bytes per 4-bit param

print(f"DistilBERT ({params/1e6:.0f}M params) — Memory by Precision:")
print(f"  FP32 (full):  {fp32_mb:>7.1f} MB")
print(f"  FP16 (half):  {fp16_mb:>7.1f} MB")
print(f"  INT8 (8-bit): {int8_mb:>7.1f} MB")
print(f"  NF4  (4-bit): {int4_mb:>7.1f} MB")

# Now scale to a 7B model
print(f"\nLlama-2-7B (7B params) — Memory by Precision:")
big_params = 7e9
print(f"  FP32 (full):  {big_params * 4 / 1e9:>5.1f} GB  <- needs A100 80GB")
print(f"  FP16 (half):  {big_params * 2 / 1e9:>5.1f} GB  <- needs A100 40GB")
print(f"  INT8 (8-bit): {big_params * 1 / 1e9:>5.1f} GB  <- fits on V100")
print(f"  NF4  (4-bit): {big_params * 0.5 / 1e9:>5.1f} GB  <- fits on T4!")

print("\nThis is why QLoRA is essential for fine-tuning large models.")
print("4-bit loading + LoRA adapters = fine-tune 7B on free Colab T4.")

In [ ]:
# =============================================================================
# DEMO: Inspecting LoRA Adapter Layers
# =============================================================================
# Let's peek inside the model to see exactly which layers have LoRA adapters
# and which are frozen. This helps you understand what target_modules controls.

print("Model Architecture — LoRA Adapter Layers:")
print("=" * 70)

for name, param in lora_model.named_parameters():
    # Only show attention-related layers (keep output manageable)
    if 'attention' in name or 'lora' in name or 'classifier' in name:
        status = "TRAINABLE" if param.requires_grad else "FROZEN"
        print(f"  [{status:9s}] {name:50s} shape={list(param.shape)}")

print(f"\n{'='*70}")
print("Notice:")
print("  - Original attention weights (q_lin.weight, v_lin.weight) are FROZEN")
print("  - LoRA adds lora_A and lora_B matrices alongside each target module")
print("  - The classifier head is also TRAINABLE (it's task-specific)")
print(f"  - k_lin (key) has NO adapter — we only targeted q and v")

## Lab: Experiment with LoRA Rank

### Your Task

Train LoRA adapters with different ranks (r=4, r=8, r=16, r=32) and compare:
- Number of trainable parameters
- Validation accuracy
- Training time

### Steps

1. Loop through ranks [4, 8, 16, 32]
2. For each rank, create a new LoRA config, apply to a fresh model, and train for 1 epoch
3. Record trainable params, accuracy, and training time
4. Create a summary table and identify the best rank

### Expected Output

- Table with 4 rows (one per rank)
- Best rank identified by accuracy

### Homework Extension

Try adding more target modules (e.g., `["q_lin", "k_lin", "v_lin", "out_lin"]`)
with your best rank. Does targeting more layers improve accuracy?

In [ ]:
# =============================================================================
# LAB: EXPERIMENT WITH LORA RANK
# =============================================================================

ranks = [4, 8, 16, 32]
lab_results = []

for r in ranks:
    # YOUR CODE: Create LoraConfig with this rank (lora_alpha = 2*r)
    config = None  # YOUR CODE

    # YOUR CODE: Load fresh base model and apply LoRA
    fresh_model = None  # YOUR CODE
    lora_model_lab = None  # YOUR CODE

    # YOUR CODE: Count trainable parameters
    trainable_count = None  # YOUR CODE

    # YOUR CODE: Create TrainingArguments (1 epoch, lr=2e-4)
    args = None  # YOUR CODE

    # YOUR CODE: Create Trainer and train
    lab_trainer = None  # YOUR CODE

    # YOUR CODE: Evaluate and record results
    eval_result = None  # YOUR CODE

    lab_results.append({
        'rank': r,
        'trainable_params': trainable_count,
        'accuracy': eval_result,
    })

# YOUR CODE: Create summary table
rank_df = None  # YOUR CODE

# Verification
if rank_df is not None:
    display(rank_df)
    best = rank_df.loc[rank_df['accuracy'].idxmax()]
    print(f"\nBest rank: r={best['rank']}, accuracy={best['accuracy']:.1%}")
    print("\nLab complete!")
else:
    print("Lab incomplete — fill in the YOUR CODE sections above")

# Summary

## What You Learned

| Technique | Parameters Trained | Memory Needed | Best For |
|-----------|-------------------|---------------|----------|
| **Full Fine-Tuning** | All (100%) | High | Small models (< 1B) |
| **LoRA** | ~1-5% | Medium | Medium models (1-7B) |
| **QLoRA** | ~1-5% (4-bit base) | Low | Large models (7B+) |

## Key Takeaways

1. **LoRA adds small adapter matrices** instead of updating all weights
2. **Rank (r)** controls adapter capacity — r=8 is a good default
3. **QLoRA** combines 4-bit quantization with LoRA for maximum memory savings
4. **Higher rank != always better** — diminishing returns after r=16
5. **LoRA adapters are tiny** — easy to save, share, and swap

## Resources

- [LoRA Paper](https://arxiv.org/abs/2106.09685) — Hu et al., 2021
- [QLoRA Paper](https://arxiv.org/abs/2305.14314) — Dettmers et al., 2023
- [PEFT Library Documentation](https://huggingface.co/docs/peft)
- [BitsAndBytes Documentation](https://huggingface.co/docs/bitsandbytes)